### **Parte 5: Proyección II**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Herramientas de Machine Learning (Scikit-Learn)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Cargar el dataset final
df = pd.read_csv('../data/processed/movies_listo_para_modelo.csv')

# Limpieza de seguridad: eliminar valores infinitos si hubo divisiones por cero en el ROI
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['roi', 'budget', 'temporada'])

print(f"Datos cargados listos para entrenar: {df.shape[0]} películas.")

In [ ]:
# 1. Definir la variable objetivo (Lo que queremos predecir)
y = df['roi']

# 2. Definir las variables predictoras (Nuestras características de la Fase 4)
features = ['budget', 'popularity', 'temporada', 'es_drama', 'es_comedia', 'es_thriller']
X = df[features]

# 3. Dividir en Entrenamiento (80%) y Prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Datos de entrenamiento: {X_train.shape[0]} | Datos de prueba: {X_test.shape[0]}")

In [ ]:
# Celda 3: Creación del Pipeline y Entrenamiento con Random Forest

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

# 1. Separar columnas por tipo para tratarlas distinto
num_features = ['budget', 'popularity', 'es_drama', 'es_comedia', 'es_thriller']
cat_features = ['temporada']

# 2. Crear el transformador de columnas (Escalar números, Codificar texto)
preprocesador = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ])

# 3. Ensamblar el Pipeline final con Random Forest
modelo_pipeline_rf = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('algoritmo', RandomForestRegressor(n_estimators=100, random_state=42))
])

# 4. ENTRENAR EL NUEVO MODELO
modelo_pipeline_rf.fit(X_train, y_train)

print("Pipeline ejecutado y Random Forest entrenado exitosamente.")

In [ ]:
y_pred_rf = modelo_pipeline_rf.predict(X_test)

mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("--- RESULTADOS RANDOM FOREST ---")
print(f"Error Cuadrático Medio (MSE): {mse_rf:.2f}")
print(f"Precisión del Modelo (R2 Score): {r2_rf:.4f}")

In [ ]:
# Extraer la importancia de cada variable (específico de Random Forest)
importancias = modelo_pipeline_rf.named_steps['algoritmo'].feature_importances_

# Crear el DataFrame para graficar
df_importancia_rf = pd.DataFrame({
    'Variable': nombres_finales,
    'Importancia': importancias
}).sort_values(by='Importancia', ascending=False)

# Graficar
plt.figure(figsize=(10, 6))
sns.barplot(x='Importancia', y='Variable', data=df_importancia_rf, palette='magma')
plt.title('¿Qué variables definen el éxito? (Importancia Random Forest)', fontsize=14)
plt.xlabel('Nivel de Importancia (0 a 1)')
plt.ylabel('Característica')
plt.tight_layout()
plt.savefig('../outputs/figures/importancia_random_forest.png')
plt.show()